# Brain Tumor Segmentation — Data Exploration

This notebook loads and explores the BraTS2020 dataset before any training takes place. The goal is to understand how multi-modal 3D MRI data is structured, verify that image-mask pairs are complete and aligned, and get a feel for tumor size and shape variation across patients — all of which will shape decisions in the preprocessing and training notebooks that follow.

## What this notebook does

1. Extracts the BraTS2020 dataset from the zip file
2. Verifies folder structure and checks that each patient has all 4 MRI modalities + a segmentation mask
3. Loads sample volumes with `nibabel` and visualizes 2D slices across modalities
4. Computes dataset-wide statistics (volume dimensions, tumor size distribution, class balance)

## Dataset

- **BraTS2020** (Training + Validation), source: [Kaggle — awsaf49/brats20-dataset-training-validation](https://www.kaggle.com/datasets/awsaf49/brats20-dataset-training-validation)
- 369 patients (training set), 4 MRI modalities per patient (T1, T1ce, T2, FLAIR), NIfTI format (`.nii`)
- Labels: background, necrotic/non-enhancing tumor core (NCR/NET), peritumoral edema (ED), GD-enhancing tumor (ET)
- License: CC0 Public Domain

## Output

- `results/figures/sample_slices.png`
- `results/figures/modality_comparison.png`
- `results/figures/tumor_size_distribution.png`

In [ ]:
# Import libraries

import os
import zipfile
import random

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\brain-tumor-segmentation-3d\repo"

PATHS = {
    # Raw zip file
    "brats_zip": os.path.join(BASE_DIR, "data", "raw", "brats20-nifti.zip"),

    # Extracted dataset folder
    "brats_dir": os.path.join(BASE_DIR, "data", "brats2020"),

    # Output folders
    "figures": os.path.join(BASE_DIR, "results", "figures"),
    "metrics": os.path.join(BASE_DIR, "results", "metrics"),
}

for key in ["figures", "metrics"]:
    os.makedirs(PATHS[key], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:12s} -> {path}")

In [ ]:
# Extract Dataset
# Run this once to extract the zip file into its target folder

def extract_zip(zip_path, target_dir, dataset_name):
    if not os.path.exists(zip_path):
        print(f"ERROR: zip not found -> {zip_path}")
        return False

    if os.path.exists(target_dir) and len(os.listdir(target_dir)) > 0:
        print(f"{dataset_name}: already extracted, skipping.")
        return True

    print(f"Extracting {dataset_name}...")
    os.makedirs(target_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(target_dir)

    print(f"{dataset_name} extracted to: {target_dir}")
    return True


extract_zip(PATHS["brats_zip"], PATHS["brats_dir"], "BraTS2020")

In [ ]:
# Verify Folder Structure
# Walk down into the extracted folder to find the actual patient directories

brats_dir = PATHS["brats_dir"]

top_level = os.listdir(brats_dir)
print(f"Top-level contents: {top_level}")

# Go one level deeper
level2_path = os.path.join(brats_dir, top_level[0])
level2 = os.listdir(level2_path)
print(f"\nInside '{top_level[0]}': {level2}")

# Go one more level deeper (expecting the actual patient folders here)
level3_path = os.path.join(level2_path, level2[0])
if os.path.isdir(level3_path):
    level3 = os.listdir(level3_path)
    print(f"\nInside '{level2[0]}': {len(level3)} items")
    print("First 5 items:")
    for item in level3[:5]:
        print(f"  {item}")

In [ ]:
# Inspect a single patient folder to confirm the 5 expected NIfTI files

training_dir = os.path.join(brats_dir, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")

# Filter out the CSV files, keep only patient folders
patient_folders = sorted([
    f for f in os.listdir(training_dir)
    if os.path.isdir(os.path.join(training_dir, f))
])

print(f"Total patient folders: {len(patient_folders)}")

sample_patient = patient_folders[0]
sample_path = os.path.join(training_dir, sample_patient)

print(f"\nFiles inside '{sample_patient}':")
for f in os.listdir(sample_path):
    file_path = os.path.join(sample_path, f)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"  {f}  ({size_mb:.1f} MB)")